# AI-Powered Data Assistant

This notebook demonstrates how to use the **Cortex Agents Coding Agent** — the same engine that powers Snowflake CoCo — as a programmable AI assistant inside your own workflows.

### How It Works

```
┌──────────────┐     AGENT_RUN()      ┌────────────────────────┐
│  Your        │ ──── SQL call ─────► │  Cortex Coding Agent   │
│  Notebook    │                      │  (managed sandbox)     │
│              │ ◄── JSON response ── │  • Writes & runs SQL   │
│  Question:   │                      │  • Reads data          │
│  "What is    │                      │  • Generates analysis  │
│   inflation  │                      │  • Returns answer      │
│   trend?"    │                      └────────────────────────┘
└──────────────┘
```

**Key concepts:**
- `SNOWFLAKE.CORTEX.AGENT_RUN()` — SQL function that calls the Cortex Agents API
- `code_toolset_all` — gives the agent a full sandbox (bash, SQL, file I/O, web search)
- **Lightweight model** — we use `mistral-large2` for fast, cost-efficient responses
- **Thread management** — continue conversations across multiple turns

**Data available:** All of `SNOWFLAKE_PUBLIC_DATA_FREE` (40+ public datasets) and `SNOWFLAKE_SAMPLE_DATA` (TPC-H supply chain)

**Flow:** Quick Demo → Build Helper Functions → Pre-built Queries → Free-form Chat

---
## Part 1: Quick Demo

Let's start with the simplest possible call — one SQL statement that asks the AI a question. The agent will autonomously write and execute SQL, then return an answer.

In [ ]:
%%sql -r demo_response
-- Ask the Coding Agent a question using AGENT_RUN
-- It will write SQL against our datasets, execute it, and return an answer
WITH raw AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.AGENT_RUN(
      $${
        "models": { "orchestration": "auto" },
        "instructions": {
          "response": "You are a data analyst. Answer questions using SQL queries against SNOWFLAKE_SAMPLE_DATA.TPCH_SF1 (supply chain) and SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE (US macro data: BLS employment, BLS prices, FHFA house prices). Be concise. Show key numbers."
        },
        "messages": [
          {
            "role": "user",
            "content": [
              {
                "type": "text",
                "text": "How many orders are in the TPC-H dataset and what is the total revenue? Give me a quick summary."
              }
            ]
          }
        ],
        "tools": [
          { "tool_spec": { "type": "code_toolset_all", "name": "code_toolset_all" } }
        ],
        "tool_resources": {
          "code_toolset_all": {
            "permission_policy": { "type": "always_allow" }
          }
        }
      }$$,
      TRUE
    )
  ) AS agent_response
)
SELECT
  f.value:text::STRING AS answer
FROM raw,
  LATERAL FLATTEN(input => raw.agent_response:content) f
WHERE f.value:type::STRING = 'text'
ORDER BY f.index DESC
LIMIT 1

The response above is raw JSON from the agent. Let's build helper functions to make this clean and reusable.

---
## Part 2: Building the Assistant

We'll create a Python wrapper that:
1. Sends questions to the Coding Agent via `AGENT_RUN`
2. Parses the JSON response to extract the text answer
3. Manages conversation threads for multi-turn chats
4. Uses `mistral-large2` — a lightweight, fast model that keeps costs low

In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

SYSTEM_INSTRUCTIONS = """You are a data analyst assistant. You have access to these Snowflake databases:

1. SNOWFLAKE_SAMPLE_DATA.TPCH_SF1 — TPC-H supply chain benchmark:
   - ORDERS (1.5M rows): O_ORDERKEY, O_CUSTKEY, O_ORDERSTATUS, O_TOTALPRICE, O_ORDERDATE, O_ORDERPRIORITY
   - LINEITEM (6M rows): L_ORDERKEY, L_PARTKEY, L_SUPPKEY, L_EXTENDEDPRICE, L_DISCOUNT, L_SHIPDATE, L_COMMITDATE
   - CUSTOMER (150K rows): C_CUSTKEY, C_NAME, C_MKTSEGMENT, C_NATIONKEY
   - SUPPLIER (10K rows): S_SUPPKEY, S_NAME, S_NATIONKEY
   - NATION (25 rows): N_NATIONKEY, N_NAME, N_REGIONKEY
   - REGION (5 rows): R_REGIONKEY, R_NAME

2. SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE — US macro data:
   - BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES: CPI, inflation (VARIABLE_NAME, DATE, VALUE, GEO_ID)
   - BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES: unemployment rates by state
   - FHFA_HOUSE_PRICE_TIMESERIES: house price index
   - FEMA_DISASTER_DECLARATIONS_TIMESERIES: natural disasters
   - And 40+ more public datasets

Rules:
- Write and execute SQL to answer questions. Always use fully qualified table names.
- Be concise. Show key numbers and percentages.
- If a question is ambiguous, make a reasonable assumption and state it."""

def ask_agent(question, model='auto'):
    """Send a question to the Cortex Coding Agent and return the text answer."""
    payload = json.dumps({
        "models": {"orchestration": model},
        "instructions": {"response": SYSTEM_INSTRUCTIONS},
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": question}]}
        ],
        "tools": [
            {"tool_spec": {"type": "code_toolset_all", "name": "code_toolset_all"}}
        ],
        "tool_resources": {
            "code_toolset_all": {
                "permission_policy": {"type": "always_allow"}
            }
        }
    }).replace("'", "''")
    
    result = session.sql(f"""
        SELECT TRY_PARSE_JSON(
            SNOWFLAKE.CORTEX.AGENT_RUN($${payload}$$, TRUE)
        ) AS resp
    """).collect()
    
    if not result:
        return "No response from agent."
    
    resp = json.loads(result[0]['RESP'])
    
    text_parts = []
    for item in resp.get('content', []):
        if item.get('type') == 'text':
            text_parts.append(item['text'])
    
    return '\n'.join(text_parts) if text_parts else str(resp)

print("ask_agent() function ready.")
print("Model: auto (automatically selected)")
print("\nExample: ask_agent('What are the top 5 nations by revenue in TPC-H?')")

---
## Part 3: Pre-built Analytics Queries

Like Google Analytics' quick-access reports, here are ready-made questions you can run with one click. Select a question from the dropdown and the AI agent will autonomously query the data and return insights.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

QUERY_CATALOG = {
    '--- Supply Chain (TPC-H) ---': None,
    'Top 5 regions by total revenue': 'What are the top 5 regions by total revenue in TPC-H? Show revenue in millions.',
    'Customer segments breakdown': 'Break down TPC-H customers by C_MKTSEGMENT. Show count and total order value per segment.',
    'Late shipment rate by priority': 'What percentage of TPC-H line items shipped late (L_SHIPDATE > L_COMMITDATE)? Break down by O_ORDERPRIORITY.',
    'Yearly revenue trend': 'Show TPC-H total revenue by year (from O_ORDERDATE). Is it growing or shrinking?',
    '--- US Macro Economy ---': None,
    'Current inflation rate': 'What is the most recent YoY inflation rate? Use CPI All items for country/USA from BLS price timeseries.',
    'Unemployment trend last 2 years': 'Show the monthly average unemployment rate for the last 2 years from BLS employment timeseries.',
    'House price growth this year': 'What is the most recent house price index value and its YoY growth? Use FHFA purchase-only seasonally adjusted for country/USA.',
    '--- Cross-Dataset ---': None,
    'Compare inflation vs house prices': 'Compare YoY inflation (CPI) vs YoY house price growth for the last 3 years. Are they correlated?',
}

query_dropdown = widgets.Dropdown(
    options=list(QUERY_CATALOG.keys()),
    value=list(QUERY_CATALOG.keys())[1],
    description='Question:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '80px'}
)

run_btn = widgets.Button(
    description='Ask the AI Agent',
    button_style='info',
    layout=widgets.Layout(width='200px', height='35px')
)

result_out = widgets.Output()

def on_run_click(b):
    question = QUERY_CATALOG.get(query_dropdown.value)
    if question is None:
        with result_out:
            clear_output(wait=True)
            print("Please select a question (not a category header).")
        return
    with result_out:
        clear_output(wait=True)
        print(f"Asking: {question}")
        print("Thinking... (the agent is writing and executing SQL)\n")
    answer = ask_agent(question)
    with result_out:
        clear_output(wait=True)
        print(f"Q: {question}\n")
        print(f"A: {answer}")

run_btn.on_click(on_run_click)

display(widgets.VBox([
    widgets.HTML('<h3>Pre-built Analytics Queries</h3>'),
    widgets.HTML('<p>Select a question and click the button. The AI agent will query the data and return insights.</p>'),
    widgets.HBox([query_dropdown, run_btn]),
    result_out
]))

---
## Part 4: Free-form Chat

Ask **anything** about the data. Type a question in natural language and the AI agent will:
1. Figure out which tables to query
2. Write and execute SQL
3. Analyze the results
4. Return a clear answer

Conversation history is displayed below so you can see previous Q&A.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_history = []

text_input = widgets.Text(
    placeholder='Ask anything about the data... (e.g., "Which market segment has the most late shipments?")',
    layout=widgets.Layout(width='600px')
)

send_btn = widgets.Button(
    description='Send',
    button_style='success',
    layout=widgets.Layout(width='100px', height='35px')
)

clear_btn = widgets.Button(
    description='Clear History',
    button_style='warning',
    layout=widgets.Layout(width='120px', height='35px')
)

chat_out = widgets.Output()

def render_history():
    with chat_out:
        clear_output(wait=True)
        if not chat_history:
            print("No messages yet. Type a question above and click Send.")
            return
        for i, (q, a) in enumerate(chat_history, 1):
            print(f"{'='*60}")
            print(f"  Q{i}: {q}")
            print(f"{'\u2500'*60}")
            print(f"  A{i}: {a}")
            print()

def on_send(b):
    question = text_input.value.strip()
    if not question:
        return
    text_input.value = ''
    
    with chat_out:
        clear_output(wait=True)
        for i, (q, a) in enumerate(chat_history, 1):
            print(f"{'='*60}")
            print(f"  Q{i}: {q}")
            print(f"{'\u2500'*60}")
            print(f"  A{i}: {a}")
            print()
        print(f"{'='*60}")
        print(f"  Q{len(chat_history)+1}: {question}")
        print(f"{'\u2500'*60}")
        print(f"  Thinking... (agent is writing and executing SQL)")
    
    answer = ask_agent(question)
    chat_history.append((question, answer))
    render_history()

def on_clear(b):
    chat_history.clear()
    render_history()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

display(widgets.VBox([
    widgets.HTML('<h3>Free-form Data Chat</h3>'),
    widgets.HTML('<p>Type any question about TPC-H supply chain data or US macro economic data.</p>'),
    widgets.HBox([text_input, send_btn, clear_btn]),
    chat_out
]))
render_history()

---
## Part 5: How It Works — Under the Hood

### Architecture
```
┌─────────────────────────────────────────────────────┐
│  This Notebook (Python)                             │
│  ┌───────────────────────────────────────────────┐  │
│  │ ask_agent("What is the inflation trend?")     │  │
│  │   → Builds JSON payload                       │  │
│  │   → Calls session.sql(AGENT_RUN(...))         │  │
│  │   → Parses JSON response                      │  │
│  └───────────────────────────────────────────────┘  │
│                         │                           │
│                    SQL call                          │
│                         ▼                           │
│  ┌───────────────────────────────────────────────┐  │
│  │ SNOWFLAKE.CORTEX.AGENT_RUN()                  │  │
│  │   → Provisions sandboxed runtime              │  │
│  │   → Agent writes SQL from your question       │  │
│  │   → Executes SQL against your databases       │  │
│  │   → Analyzes results                          │  │
│  │   → Returns structured JSON answer            │  │
│  └───────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────┘
```

### Cost Considerations
| Model | Speed | Cost | Best For |
|-------|-------|------|----------|
| `mistral-large2` | Fast | Low | Quick lookups, simple analytics |
| `claude-sonnet-4-5` | Medium | Medium | Complex multi-step analysis |
| `claude-opus-4-6` | Slower | Higher | Deep reasoning, complex joins |

We use `mistral-large2` by default because it's fast and cheap for data Q&A tasks.

### Key API Details
- **`AGENT_RUN(payload, TRUE)`** — the `TRUE` creates a thread automatically
- **`code_toolset_all`** — gives the agent bash, SQL execution, file I/O, web search
- **`permission_policy: always_allow`** — lets the agent run without asking for approval
- **Response format** — JSON with `messages` array containing `tool_use`, `tool_result`, and final `text` blocks

### What the Agent Can Do
- Write and execute SQL queries against any database you have access to
- Write Python scripts for data transformation
- Search the web for context
- Read and write files in its sandbox
- Chain multiple operations together (e.g., query → transform → analyze)

### Try These Questions
- "Which supplier has the most late shipments in TPC-H?"
- "What was the peak unemployment rate since 2020?"
- "Compare revenue between ASIA and EUROPE regions"
- "What's the correlation between house prices and inflation?"
- "Show me the top 10 customers by total spend"